# 🔄 End-to-End Data Integration and ETL Pipeline using PySpark

## 📌 Overview

In this project, we build an end-to-end **ETL pipeline using Apache Spark and PySpark**.

The goal is to demonstrate how Spark can be used to extract data from different sources, transform datasets using the DataFrame API and Spark SQL, integrate data through joins and aggregations, and finally load the processed results into analytical storage formats.

This project reflects a typical **Data Engineering workflow** — from raw data ingestion to transformation, integration, SQL-based analysis, and saving processed outputs for downstream analytics.

---

## 🎯 Objectives

- Load structured data into Spark DataFrames
- Perform column transformations using the **PySpark DataFrame API**
- Apply data cleaning operations such as renaming and dropping columns
- Register temporary views and run **Spark SQL queries**
- Create and use a custom **UDF** for feature transformation
- Join multiple datasets for data integration
- Perform aggregations for analytical reporting
- Save transformed results in **CSV** and **Parquet** formats

---

## 🛠️ Technologies Used

- **Apache Spark**
- **PySpark**
- **Spark DataFrame API**
- **Spark SQL**
- **User Defined Functions (UDF)**
- **CSV**
- **Parquet**

---

## 🧱 ETL Workflow

This project follows a standard ETL process:

1. **Extract** — load raw structured data into Spark  
2. **Transform** — clean, enrich, join, and aggregate datasets  
3. **Load** — save processed results for analytics and reporting 

## 💡 Business Context

Many companies collect data from multiple operational systems such as sales, product, employee, or vehicle datasets.
Before this data can be used for reporting or machine learning, it must be cleaned, transformed, joined, and stored in an analytics-ready format.

This project demonstrates how PySpark can support this process at scale.

## 1. Import + Spark Initialization

In this section, we initialize the Spark environment.

We use:
- **findspark** to locate the Spark installation
- **SparkContext** to connect to the cluster
- **SparkSession** to work with DataFrames and SQL

In [104]:
# Initialize Spark environment

import findspark
findspark.init()

# Spark imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round, sum, avg, count, to_date, pandas_udf, year, quarter, when, lit

# Other libraries
import pandas as pd

# Create Spark Session
spark = SparkSession \
    .builder \
    .appName("Spark ETL Data Integration Pipeline") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark session initialized successfully")
print(f"Spark version: {spark.version}")

Spark session initialized successfully
Spark version: 3.5.1


## 2. Data Loading

In this section, we load raw transactional datasets into Spark DataFrames.

- **transactions** dataset contains purchase records  
- **transaction_details** dataset contains additional information for each transaction  

These datasets will be used later for data integration and analysis.

In [105]:
# Load datasets (same directory as notebook)

transactions_df = spark.read.csv(
    "data/transactions.csv.csv",
    header=True,
    inferSchema=True
)

details_df = spark.read.csv(
    "data/transaction_details.csv",
    header=True,
    inferSchema=True
)

# Preview data

transactions_df.show(5)
details_df.show(5)

+-----------+-----------+------+-----------+--------+
|customer_id|date_column|amount|description|location|
+-----------+-----------+------+-----------+--------+
|          1|   1/1/2022|  5000| Purchase A| Store A|
|          2|  15/2/2022|  1200| Purchase B| Store B|
|          3|  20/3/2022|   800| Purchase C| Store C|
|          4|  10/4/2022|  3000| Purchase D| Store D|
|          5|   5/5/2022|  6000| Purchase E| Store E|
+-----------+-----------+------+-----------+--------+
only showing top 5 rows

+-----------+----------------+-----+------+
|customer_id|transaction_date|value| notes|
+-----------+----------------+-----+------+
|          1|        1/1/2022| 1500|Note 1|
|          2|       15/2/2022| 2000|Note 2|
|          3|       20/3/2022| 1000|Note 3|
|          4|       10/4/2022| 2500|Note 4|
|          5|        5/5/2022| 1800|Note 5|
+-----------+----------------+-----+------+
only showing top 5 rows



The datasets are successfully loaded into Spark DataFrames and previewed for initial inspection.

## 3. Schema Inspection

Before transforming the data, we first inspect the structure of both DataFrames.

This helps us identify:
- column names that should be standardized
- date columns stored as strings
- columns that need clearer business-friendly names
- schema differences before joining the datasets

In [106]:
# Inspect schemas before transformation

transactions_df.printSchema()
details_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- date_column: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- location: string (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- value: integer (nullable = true)
 |-- notes: string (nullable = true)



The schema inspection reveals differences in column names and structures, which will be addressed in the transformation step.

### Transformation Plan

Based on the initial schema inspection, we will apply the following changes:

- Rename `date_column` to `transaction_date` in the transactions dataset
- Rename `value` to `transaction_value` in the details dataset
- Convert `transaction_date` columns from string format to proper date type
- Keep column names consistent before joining the datasets

## 4. Data Cleaning and Transformation

In this section, we clean and prepare the datasets for further processing.

The main steps include:
- Renaming columns for consistency
- Converting date columns to proper date format
- Ensuring consistent schema across datasets

These transformations are essential before performing data integration.

In [107]:
# Rename columns for consistency

transactions_df = transactions_df.withColumnRenamed("date_column", "transaction_date")
details_df = details_df.withColumnRenamed("value", "transaction_value")

In [108]:
# Convert date columns to date type

transactions_df = transactions_df.withColumn(
    "transaction_date",
    to_date(col("transaction_date"), "d/M/yyyy")
)

details_df = details_df.withColumn(
    "transaction_date",
    to_date(col("transaction_date"), "d/M/yyyy")
)

In [109]:
# Inspect schemas after transformation

transactions_df.printSchema()
details_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- amount: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- location: string (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- transaction_value: integer (nullable = true)
 |-- notes: string (nullable = true)



The schema is now standardized, with consistent column names and proper date data types.

In [110]:
# Preview cleaned data

transactions_df.show(5)
details_df.show(5)

+-----------+----------------+------+-----------+--------+
|customer_id|transaction_date|amount|description|location|
+-----------+----------------+------+-----------+--------+
|          1|      2022-01-01|  5000| Purchase A| Store A|
|          2|      2022-02-15|  1200| Purchase B| Store B|
|          3|      2022-03-20|   800| Purchase C| Store C|
|          4|      2022-04-10|  3000| Purchase D| Store D|
|          5|      2022-05-05|  6000| Purchase E| Store E|
+-----------+----------------+------+-----------+--------+
only showing top 5 rows

+-----------+----------------+-----------------+------+
|customer_id|transaction_date|transaction_value| notes|
+-----------+----------------+-----------------+------+
|          1|      2022-01-01|             1500|Note 1|
|          2|      2022-02-15|             2000|Note 2|
|          3|      2022-03-20|             1000|Note 3|
|          4|      2022-04-10|             2500|Note 4|
|          5|      2022-05-05|             1800|Note

The datasets are now cleaned and properly formatted, ready for data integration and further analysis.

## 5. Data Integration

In this section, we join the cleaned transactional datasets into a single integrated DataFrame.

The datasets are joined using:
- `customer_id`
- `transaction_date`

This step combines purchase records with additional transaction details and prepares the data for analytical aggregations.

In [111]:
# Join datasets using customer_id and transaction_date

joined_df = transactions_df.join(
    details_df,
    on=["customer_id", "transaction_date"],
    how="inner"
)

# Preview integrated data

joined_df.show(10)
joined_df.printSchema()

+-----------+----------------+------+-----------+--------+-----------------+-------+
|customer_id|transaction_date|amount|description|location|transaction_value|  notes|
+-----------+----------------+------+-----------+--------+-----------------+-------+
|          1|      2022-01-01|  5000| Purchase A| Store A|             1500| Note 1|
|          2|      2022-02-15|  1200| Purchase B| Store B|             2000| Note 2|
|          3|      2022-03-20|   800| Purchase C| Store C|             1000| Note 3|
|          4|      2022-04-10|  3000| Purchase D| Store D|             2500| Note 4|
|          5|      2022-05-05|  6000| Purchase E| Store E|             1800| Note 5|
|          6|      2022-06-10|  4500| Purchase F| Store F|             1200| Note 6|
|          7|      2022-07-15|   200| Purchase G| Store G|              700| Note 7|
|          8|      2022-08-20|  3500| Purchase H| Store H|             3000| Note 8|
|          9|      2022-09-25|   700| Purchase I| Store I|       

The datasets are successfully joined, combining transaction records with additional details into a single structured DataFrame.

In [112]:
# Check row counts before and after join

print("Transactions rows:", transactions_df.count())
print("Details rows:", details_df.count())
print("Joined rows:", joined_df.count())

Transactions rows: 100
Details rows: 100
Joined rows: 100


The row count remains consistent after the join, indicating that all records were successfully matched without data loss.

## 6. Feature Engineering

In this section, we create additional analytical columns from the integrated dataset.

These features will help us better analyze transaction behavior by year, quarter, value difference, and transaction value category.

The main steps include:
- Extracting year and quarter from the transaction date
- Creating a difference column between transaction amount and transaction value
- Adding a high-value transaction flag
### Feature Engineering Logic

To prepare the integrated dataset for analysis, we create several new columns:

- `year` — extracted from `transaction_date` for yearly aggregation
- `quarter` — extracted from `transaction_date` for quarterly analysis
- `amount_difference` — helps identify inconsistencies or differences between data sources
- `high_value_transaction` — flag indicating whether the transaction amount is greater than 5000

In [113]:
# Create additional analytical features

featured_df = joined_df.withColumn(
    "year",
    year(col("transaction_date"))
).withColumn(
    "quarter",
    quarter(col("transaction_date"))
).withColumn(
    "amount_difference",
    col("amount") - col("transaction_value")
).withColumn(
    "high_value_transaction",
    when(col("amount") > 5000, lit("Yes")).otherwise(lit("No"))
)

# Preview featured data

featured_df.show(10)
featured_df.printSchema()

+-----------+----------------+------+-----------+--------+-----------------+-------+----+-------+-----------------+----------------------+
|customer_id|transaction_date|amount|description|location|transaction_value|  notes|year|quarter|amount_difference|high_value_transaction|
+-----------+----------------+------+-----------+--------+-----------------+-------+----+-------+-----------------+----------------------+
|          1|      2022-01-01|  5000| Purchase A| Store A|             1500| Note 1|2022|      1|             3500|                    No|
|          2|      2022-02-15|  1200| Purchase B| Store B|             2000| Note 2|2022|      1|             -800|                    No|
|          3|      2022-03-20|   800| Purchase C| Store C|             1000| Note 3|2022|      1|             -200|                    No|
|          4|      2022-04-10|  3000| Purchase D| Store D|             2500| Note 4|2022|      2|              500|                    No|
|          5|      2022-05-

The dataset is now fully prepared for analytical aggregations and business insights by customer, year, and quarter.

The newly created features provide additional analytical value:

- `amount_difference` highlights discrepancies between transaction sources, showing both positive and negative differences.
- `high_value_transaction` flags transactions above 5000, allowing quick identification of high-value purchases.

These features enhance data quality checks and enable deeper business analysis.

## 7. Data Aggregations (DataFrame API)

This section uses the PySpark DataFrame API to compute analytical metrics.

In this section, we analyze the enriched dataset by computing key business metrics.

These aggregations help answer important questions such as:
- How much each customer spends
- Revenue distribution across time (year and quarter)
- Total transaction value trends

We perform aggregations at different levels:
- Customer level
- Year level
- Quarter level

 We calculate the following key metrics:

- **Total transaction amount per customer** — to understand customer spending behavior
- **Total transaction amount per year** — to identify yearly revenue trends
- **Total transaction amount per quarter** — to analyze seasonal patterns
- **High-value transaction distribution** — to evaluate the share of large transactions
- **Average transaction amount per quarter** — to measure typical transaction size over time

### a. Total transaction amount per customer

This metric shows how much each customer has spent in total.
It helps identify high-value customers.


In [114]:
# Total transaction amount per customer

total_per_customer = featured_df.groupBy("customer_id") \
    .agg(sum("amount").alias("total_amount"))

total_per_customer.show()

+-----------+------------+
|customer_id|total_amount|
+-----------+------------+
|         31|        3200|
|         85|        1800|
|         65|         700|
|         53|         700|
|         78|        1500|
|         34|        1200|
|         81|        5500|
|         28|        2600|
|         76|        2600|
|         26|         900|
|         27|        4200|
|         44|        1000|
|         12|         900|
|         91|        3200|
|         22|        1200|
|         93|        5500|
|         47|         700|
|          1|        5000|
|         52|        2600|
|         13|        4800|
+-----------+------------+
only showing top 20 rows



This aggregation highlights customer spending patterns.

For example, some customers reach total spending above 5000, while others remain below 1000, indicating clear differences in customer value.

### b. Total transaction amount per year

This aggregation highlights overall revenue trends over time.

In [115]:
# Total transaction amount per year

total_per_year = featured_df.groupBy("year") \
    .agg(sum("amount").alias("yearly_total"))

total_per_year.show()

+----+------------+
|year|yearly_total|
+----+------------+
|2025|       25700|
|2027|       25700|
|2023|       28100|
|2022|       29800|
|2026|       25700|
|2029|       25700|
|2030|        9500|
|2028|       25700|
|2024|       25700|
+----+------------+



This aggregation reveals revenue differences across years.

For example, the total transaction amount reaches around 29,800 in some years, while dropping to approximately 9,500 in others, indicating fluctuations in performance over time.

### c. Total transaction amount per quarter

This helps analyze seasonal patterns and fluctuations within a year.

In [116]:
# Total transaction amount per quarter

total_per_quarter = featured_df.groupBy("year", "quarter") \
    .agg(sum("amount").alias("quarterly_total"))

total_per_quarter.show()

+----+-------+---------------+
|year|quarter|quarterly_total|
+----+-------+---------------+
|2029|      3|           9700|
|2029|      2|           4800|
|2027|      1|           6900|
|2024|      3|           9700|
|2030|      2|           2600|
|2027|      3|           9700|
|2028|      1|           6900|
|2029|      1|           6900|
|2022|      2|          13500|
|2026|      2|           4800|
|2022|      3|           4400|
|2024|      2|           4800|
|2030|      1|           6900|
|2023|      3|           9700|
|2026|      3|           9700|
|2023|      2|           4800|
|2029|      4|           4300|
|2025|      3|           9700|
|2023|      4|           4300|
|2028|      3|           9700|
+----+-------+---------------+
only showing top 20 rows



This aggregation highlights seasonal patterns in transaction activity across different quarters.

From the sample output, we can observe that certain quarters (e.g., Q3) frequently show higher total transaction amounts (around 9,700), while others (e.g., Q2 or Q4) sometimes have lower values (around 2,600–4,800).

This suggests potential seasonal fluctuations in customer spending behavior, which can be useful for planning promotions and resource allocation.

### d. High-value transaction distribution

This metric shows how many transactions fall into high-value vs normal categories.

In [117]:
# Count high-value vs normal transactions

high_value_stats = featured_df.groupBy("high_value_transaction") \
    .agg(count("*").alias("transaction_count"))

high_value_stats.show()

+----------------------+-----------------+
|high_value_transaction|transaction_count|
+----------------------+-----------------+
|                    No|               92|
|                   Yes|                8|
+----------------------+-----------------+



Most transactions are low-value.

Only 8% are high-value, indicating that large purchases are relatively rare.

This insight can help target strategies to increase high-value transactions.

### e. Average transaction amount per quarter

This metric represents the typical transaction size over time.

In [118]:
# Average transaction value per quarter (rounded)

avg_per_quarter = featured_df.groupBy("year", "quarter") \
    .agg(round(avg("amount"), 2).alias("avg_amount"))

avg_per_quarter.orderBy("year", "quarter").show()

+----+-------+----------+
|year|quarter|avg_amount|
+----+-------+----------+
|2022|      1|   2333.33|
|2022|      2|    4500.0|
|2022|      3|   1466.67|
|2022|      4|   1633.33|
|2023|      1|    3100.0|
|2023|      2|    1600.0|
|2023|      3|   3233.33|
|2023|      4|   1433.33|
|2024|      1|    2300.0|
|2024|      2|    1600.0|
|2024|      3|   3233.33|
|2024|      4|   1433.33|
|2025|      1|    2300.0|
|2025|      2|    1600.0|
|2025|      3|   3233.33|
|2025|      4|   1433.33|
|2026|      1|    2300.0|
|2026|      2|    1600.0|
|2026|      3|   3233.33|
|2026|      4|   1433.33|
+----+-------+----------+
only showing top 20 rows



Average transaction size varies across quarters.

For example, Q2 shows higher values (up to 4500), while Q3–Q4 are generally lower.

This indicates changes in customer spending behavior throughout the year.

## 8. Spark SQL Analytics Layer (SQL API)

This section uses Spark SQL to perform analytical queries on the dataset.

In this section, we register the enriched DataFrame as a temporary SQL view and perform analytical queries using Spark SQL.

This demonstrates how Spark DataFrames can be queried using SQL syntax, which is commonly used in data engineering and analytics workflows.

### a. Create temporary SQL view

We first register the enriched DataFrame as a temporary SQL view.  
This allows us to run SQL queries directly on the processed dataset.

In [129]:
# Create temporary SQL view

featured_df.createOrReplaceTempView("transactions_view")

This enables analysts to use familiar SQL syntax to explore the data and perform complex queries more easily.

### b. Total revenue by year

This query calculates total transaction revenue for each year using Spark SQL.  
It helps identify yearly revenue trends.

In [130]:
# Total revenue by year using Spark SQL

spark.sql("""
SELECT
    year,
    SUM(amount) AS total_revenue
FROM transactions_view
GROUP BY year
ORDER BY year
""").show()

+----+-------------+
|year|total_revenue|
+----+-------------+
|2022|        29800|
|2023|        28100|
|2024|        25700|
|2025|        25700|
|2026|        25700|
|2027|        25700|
|2028|        25700|
|2029|        25700|
|2030|         9500|
+----+-------------+



Revenue is relatively stable across most years (~25,700), with some variations.

Some years show higher values, while others are noticeably lower.

This may indicate fluctuations in demand or differences in data distribution.

### c. Top customers by total spending

This query identifies customers with the highest total spending.  
It helps detect key customers who contribute most to overall revenue.

In [131]:
# Top customers by total spending using Spark SQL

spark.sql("""
SELECT
    customer_id,
    SUM(amount) AS total_spent
FROM transactions_view
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 10
""").show()

+-----------+-----------+
|customer_id|total_spent|
+-----------+-----------+
|          5|       6000|
|         57|       5500|
|         93|       5500|
|         81|       5500|
|         69|       5500|
|         45|       5500|
|         21|       5500|
|         33|       5500|
|          1|       5000|
|         13|       4800|
+-----------+-----------+



Top customers show similar high spending (~5,500–6,000).

This indicates a group of valuable customers who contribute significantly to revenue.

They can be targeted with loyalty programs or special offers.

### d. Adjusted transaction value

This query calculates an adjusted transaction amount by applying a sample 2% processing fee.  
It demonstrates how Spark SQL expressions can be used to create calculated business metrics.

In [132]:
# Calculate adjusted transaction value using Spark SQL expression

spark.sql("""
SELECT
    customer_id,
    amount,
    ROUND(amount * 1.02, 2) AS amount_with_processing_fee
FROM transactions_view
ORDER BY amount DESC
LIMIT 10
""").show()

+-----------+------+--------------------------+
|customer_id|amount|amount_with_processing_fee|
+-----------+------+--------------------------+
|          5|  6000|                   6120.00|
|         21|  5500|                   5610.00|
|         93|  5500|                   5610.00|
|         57|  5500|                   5610.00|
|         33|  5500|                   5610.00|
|         45|  5500|                   5610.00|
|         69|  5500|                   5610.00|
|         81|  5500|                   5610.00|
|          1|  5000|                   5100.00|
|         13|  4800|                   4896.00|
+-----------+------+--------------------------+



This shows how a 2% fee increases transaction values.

Even small percentage changes (e.g., 6000 → 6120) can impact total revenue.

Useful for modeling fees, commissions, or costs.

### e. Optional UDF example

A custom UDF can also be registered and used inside Spark SQL.  
In some local Windows environments, Python UDF execution may require additional Spark, Python, or Hadoop configuration.

In [133]:
# Optional: Custom UDF example for Spark SQL
# In some local Windows environments, Python UDF execution may fail due to Spark/Python/Hadoop configuration.

# def apply_processing_fee(amount):
#     return amount * 1.02 if amount is not None else None

# spark.udf.register("apply_processing_fee", apply_processing_fee, "double")

# spark.sql("""
# SELECT
#     customer_id,
#     amount,
#     apply_processing_fee(amount) AS amount_with_processing_fee
# FROM transactions_view
# ORDER BY amount DESC
# LIMIT 10
# """).show()

This optional example demonstrates how custom UDFs can be used in Spark SQL for advanced transformations.

## 9. Data Storage

In this section, we save the processed and aggregated datasets for further use.

The data is stored in efficient formats for analytics and future processing.

We save:
- Aggregated datasets
- Filtered datasets

In [79]:
# Save datasets as CSV files

featured_df.toPandas().to_csv("output/featured_transactions.csv", index=False)
total_per_customer.toPandas().to_csv("output/total_per_customer.csv", index=False)
total_per_year.toPandas().to_csv("output/total_per_year.csv", index=False)
total_per_quarter.toPandas().to_csv("output/total_per_quarter.csv", index=False)
avg_per_quarter.toPandas().to_csv("output/avg_per_quarter.csv", index=False)

print("Datasets successfully saved to the output folder")

Datasets successfully saved to the output folder


For local portfolio execution, the final datasets are exported as CSV files.

In a production Spark environment, the same outputs can be written to distributed storage such as HDFS or cloud storage in Parquet format.

⚠️ Note: For large-scale datasets, it is recommended to use Spark native methods (e.g., write.csv or write.parquet) instead of converting to Pandas.


#### 9.1. Parquet Storage (Data Lake Layer)

In a real-world data engineering environment, datasets are typically stored in **Parquet format**.

Parquet is a columnar storage format that provides:
- Better compression
- Faster query performance
- Efficient integration with big data tools (Spark, Hive, etc.)

⚠️ Note: On some local Windows environments, Spark write operations may fail due to missing Hadoop configuration.  
For this reason, this block is provided as a reference for production usage.


In [60]:
# Save datasets in Parquet format (production-style)

# Uncomment when running in a proper Spark/Hadoop environment

# featured_df.write.mode("overwrite").parquet("output/featured_transactions_parquet")
# total_per_customer.write.mode("overwrite").parquet("output/total_per_customer_parquet")
# total_per_year.write.mode("overwrite").parquet("output/total_per_year_parquet")
# total_per_quarter.write.mode("overwrite").parquet("output/total_per_quarter_parquet")
# avg_per_quarter.write.mode("overwrite").parquet("output/avg_per_quarter_parquet")

#### 9.2. Hive Tables (Data Warehouse Layer)

In [62]:
# Save to Hive table (Data Warehouse layer)

# Uncomment when Hive support is available

# total_per_customer.write.mode("overwrite").saveAsTable("customer_totals")

#### 9.3. HDFS / Parquet (Filtered Data Export)

In [63]:
# Save filtered dataset (HDFS-style storage)

# filtered_df.write.mode("overwrite").parquet("output/filtered_data_parquet")

#### 9.4. Spark CSV Export (Interoperability)

In [64]:
# Save using Spark CSV writer

# total_per_year.write.mode("overwrite").csv("output/total_value_per_year_csv", header=True)

These examples replicate common data storage patterns used in real-world data engineering pipelines:

- Hive tables — for structured data warehousing and SQL analytics  
- Parquet files — for scalable distributed processing  
- CSV (Spark writer) — for data exchange between systems  

⚠️ These operations are environment-dependent and may require Hadoop or Hive configuration.

## 10. Conclusion and Business Insights

All datasets have been successfully **processed**, **validated**, and prepared for **downstream analytics**.

This project demonstrates a complete **end-to-end ETL and analytics pipeline** built using **Apache Spark**:

- **Data ingestion** from raw sources  
- **Data cleaning and transformation**  
- **Data integration** across multiple datasets  
- **Feature engineering** for analytical use cases  
- **Aggregation** of key business metrics  
- **Data storage** in multiple formats and layers  

The pipeline reflects real-world **data engineering practices**, including:

- **Scalable data processing**  
- **Multi-layer storage architecture**  
- **Separation between development and production environments**

It is designed to simulate a **real-world scenario** where data flows from **raw ingestion** to **analytics-ready datasets**, supporting **business insights** and **decision-making**.

### 1️⃣ Technical Summary

- Built an end-to-end **ETL pipeline** using **Apache Spark**
- Integrated multiple datasets using common keys  
- Applied data cleaning and schema standardization  
- Created analytical features for time-based and value-based analysis  
- Performed aggregations to derive business metrics  
- Implemented multi-format data storage (CSV, Parquet, Hive-style)  

---

### 2️⃣ Business Insights

- **Customer spending behavior varies significantly**, indicating different customer segments  
- **High-value transactions (>5000)** represent a smaller share of transactions but contribute disproportionately to total revenue  
- **Quarterly trends show fluctuations** in transaction volumes and average values over time  
- The **difference between transaction amount and transaction value** suggests possible discounts, adjustments, or inconsistencies between systems  
- Some customers consistently generate **higher total transaction volumes**, making them key contributors to business performance  

---

### 3️⃣ Final Conclusion

- This project demonstrates how **raw transactional data** can be transformed into **structured, analytics-ready datasets** using **Apache Spark**  
- The pipeline simulates a **real-world data engineering workflow**, enabling scalable data processing, integration, and analysis  
- Such solutions support **business decision-making** by providing clear insights into **customer behavior** and **transaction trends**


## 11. Stop Spark Session

In the final step, we gracefully stop the Spark session to release allocated resources.

This is an important best practice in real-world data engineering workflows, ensuring efficient resource management and proper pipeline completion.



In [134]:
# Stop Spark session

spark.stop()

print("Spark session stopped")

Spark session stopped
